# Module 12 Lab - Ethics, Fairness, and Bias in ML**Objective:** To understand how machine learning models can inherit and amplify societal biases, how to measure this bias using fairness metrics, and to think critically about the ethical implications of deploying ML systems.**In this lab, you will train a model on a real-world dataset and audit it for fairness across different demographic groups.**

## Part 1: What is Algorithmic Bias?**Concept:** Machine learning models learn from data. If the data reflects existing societal biases, the model will learn those biases. An "unbiased" algorithm trained on biased data will produce a biased model. This can lead to systems that are systematically unfair to certain groups of people.**Sources of Bias:***   **Historical Bias:** The data reflects a world with historical injustices (e.g., past hiring data may show fewer women in leadership roles).*   **Measurement Bias:** The way we collect or measure data is flawed (e.g., using arrest records as a proxy for crime, which can be influenced by policing patterns).*   **Representation Bias:** The data underrepresents certain groups, so the model doesn't learn to perform well for them.**Problem:** We will use the "Adult" dataset, which is used to predict whether an individual's income is greater than $50k/year. It contains sensitive attributes like `sex` and `race`, which we can use to audit our model for bias.

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score

# Load the data
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data'
columns = ['age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income']
df = pd.read_csv(url, header=None, names=columns, sep=',\s*', engine='python', na_values='?')

# Data Cleaning
df.dropna(inplace=True)
df['income'] = df['income'].map({'<=50K': 0, '>50K': 1})

X = df.drop('income', axis=1)
y = df['income']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Create a preprocessing pipeline
numeric_features = X.select_dtypes(include='number').columns
categorical_features = X.select_dtypes(exclude='number').columns

preprocessor = make_column_transformer(
    (StandardScaler(), numeric_features),
    (OneHotEncoder(handle_unknown='ignore'), categorical_features))

# Train a baseline model
model = make_pipeline(preprocessor, LogisticRegression(max_iter=1000))
model.fit(X_train, y_train)

print(f"Overall model accuracy: {model.score(X_test, y_test):.2%}")

<>:12: SyntaxWarning: invalid escape sequence '\s'
<>:12: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_1211/2471831741.py:12: SyntaxWarning: invalid escape sequence '\s'
  df = pd.read_csv(url, header=None, names=columns, sep=',\s*', engine='python', na_values='?')


Overall model accuracy: 84.61%


## Part 2: Auditing the Model for FairnessHigh overall accuracy can hide poor performance on specific subgroups. We need to audit the model by comparing its performance across sensitive attributes like `sex`.**Concept: Group Fairness**One common fairness goal is to ensure the model works equally well for different groups. We can measure this by calculating metrics for each group separately.**Your Task:** Create a function to calculate accuracy for different subgroups and then use it to compare the model's performance for males and females.

In [3]:
def get_subgroup_accuracy(model, X_test, y_test, subgroup_column, subgroup_value):
    """Calculates accuracy for a specific subgroup of the test data."""
    # 1. Create a boolean mask to select the subgroup from X_test
    subgroup_mask = X_test[subgroup_column] == subgroup_value

    # 2. Select the subgroup data
    X_subgroup = X_test[subgroup_mask]
    y_subgroup = y_test[subgroup_mask]

    # 3. Calculate and return the model's score on this subgroup
    return model.score(X_subgroup, y_subgroup)

# 4. Calculate accuracy for males and females
acc_male = get_subgroup_accuracy(model, X_test, y_test, 'sex', 'Male')
acc_female = get_subgroup_accuracy(model, X_test, y_test, 'sex', 'Female')

print(f"Accuracy for Males: {acc_male:.2%}")
print(f"Accuracy for Females: {acc_female:.2%}")

Accuracy for Males: 81.20%
Accuracy for Females: 91.81%


### Task 2: Deeper Dive with a Confusion MatrixAccuracy alone doesn't tell the whole story. Let's look at the types of errors the model makes for each group.**Your Task:** Calculate and compare the **False Positive Rate (FPR)** and **False Negative Rate (FNR)** for males and females.*   **FPR:** `FP / (FP + TN)` - The percentage of people who did NOT have high income but were incorrectly predicted to have high income.*   **FNR:** `FN / (FN + TP)` - The percentage of people who DID have high income but were incorrectly predicted to have low income.

In [4]:
from sklearn.metrics import confusion_matrix
def get_rates(model, X_test, y_test, subgroup_column, subgroup_value):
    subgroup_mask = X_test[subgroup_column] == subgroup_value
    X_subgroup = X_test[subgroup_mask]
    y_subgroup = y_test[subgroup_mask]

    y_pred_subgroup = model.predict(X_subgroup)
    tn, fp, fn, tp = confusion_matrix(y_subgroup, y_pred_subgroup).ravel()

    fpr = fp / (fp + tn)
    fnr = fn / (fn + tp)
    return fpr, fnr
# --- ENTER YOUR CODE HERE ---
# 1. Calculate the rates for males and females
fpr_male, fnr_male = get_rates(model, X_test, y_test, 'sex', 'Male')
fpr_female, fnr_female = get_rates(model, X_test, y_test, 'sex', 'Female')
print(f"Male - False Positive Rate: {fpr_male:.2%}, False Negative Rate: {fnr_male:.2%}")
print(f"Female - False Positive Rate: {fpr_female:.2%}, False Negative Rate: {fnr_female:.2%}")

Male - False Positive Rate: 10.26%, False Negative Rate: 37.80%
Female - False Positive Rate: 2.81%, False Negative Rate: 47.84%


In [5]:
from sklearn.metrics import precision_score, recall_score

def get_precision_recall(model, X_test, y_test, subgroup_column, subgroup_value):
    """Calculates precision and recall for a specific subgroup of the test data."""
    subgroup_mask = X_test[subgroup_column] == subgroup_value
    X_subgroup = X_test[subgroup_mask]
    y_subgroup = y_test[subgroup_mask]

    y_pred_subgroup = model.predict(X_subgroup)

    precision = precision_score(y_subgroup, y_pred_subgroup)
    recall = recall_score(y_subgroup, y_pred_subgroup)
    return precision, recall

# Calculate precision and recall for males and females
prec_male, rec_male = get_precision_recall(model, X_test, y_test, 'sex', 'Male')
prec_female, rec_female = get_precision_recall(model, X_test, y_test, 'sex', 'Female')

print(f"Male - Precision: {prec_male:.2%}, Recall: {rec_male:.2%}")
print(f"Female - Precision: {prec_female:.2%}, Recall: {rec_female:.2%}")

Male - Precision: 73.15%, Recall: 62.20%
Female - Precision: 71.54%, Recall: 52.16%


In [6]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score

def get_subgroup_accuracy(model, X_test, y_test, subgroup_column, subgroup_value):
    """Calculates accuracy for a specific subgroup of the test data."""
    subgroup_mask = X_test[subgroup_column] == subgroup_value
    X_subgroup = X_test[subgroup_mask]
    y_subgroup = y_test[subgroup_mask]
    return model.score(X_subgroup, y_subgroup)

def get_rates(model, X_test, y_test, subgroup_column, subgroup_value):
    subgroup_mask = X_test[subgroup_column] == subgroup_value
    X_subgroup = X_test[subgroup_mask]
    y_subgroup = y_test[subgroup_mask]

    y_pred_subgroup = model.predict(X_subgroup)
    tn, fp, fn, tp = confusion_matrix(y_subgroup, y_pred_subgroup).ravel()

    # Handle division by zero for groups where (fp + tn) or (fn + tp) might be zero
    fpr = fp / (fp + tn) if (fp + tn) != 0 else 0
    fnr = fn / (fn + tp) if (fn + tp) != 0 else 0
    return fpr, fnr

def get_precision_recall(model, X_test, y_test, subgroup_column, subgroup_value):
    """Calculates precision and recall for a specific subgroup of the test data."""
    subgroup_mask = X_test[subgroup_column] == subgroup_value
    X_subgroup = X_test[subgroup_mask]
    y_subgroup = y_test[subgroup_mask]

    y_pred_subgroup = model.predict(X_subgroup)

    # Handle cases where precision or recall might be undefined due to no positive predictions or no actual positives
    precision = precision_score(y_subgroup, y_pred_subgroup, zero_division=0)
    recall = recall_score(y_subgroup, y_pred_subgroup, zero_division=0)
    return precision, recall


# Print a summary of all calculated metrics

print("\n--- Fairness Metrics Summary ---")
print(f"Overall model accuracy: {model.score(X_test, y_test):.2%}")
print("\n--- Subgroup Metrics (by Sex) ---")

# Accuracy
acc_male = get_subgroup_accuracy(model, X_test, y_test, 'sex', 'Male')
acc_female = get_subgroup_accuracy(model, X_test, y_test, 'sex', 'Female')
print(f"Accuracy for Males: {acc_male:.2%}")
print(f"Accuracy for Females: {acc_female:.2%}")

# False Positive Rate (FPR) and False Negative Rate (FNR)
fpr_male, fnr_male = get_rates(model, X_test, y_test, 'sex', 'Male')
fpr_female, fnr_female = get_rates(model, X_test, y_test, 'sex', 'Female')
print(f"Male - False Positive Rate: {fpr_male:.2%}, False Negative Rate: {fnr_male:.2%}")
print(f"Female - False Positive Rate: {fpr_female:.2%}, False Negative Rate: {fnr_female:.2%}")

# Precision and Recall
prec_male, rec_male = get_precision_recall(model, X_test, y_test, 'sex', 'Male')
prec_female, rec_female = get_precision_recall(model, X_test, y_test, 'sex', 'Female')
print(f"Male - Precision: {prec_male:.2%}, Recall: {rec_male:.2%}")
print(f"Female - Precision: {prec_female:.2%}, Recall: {rec_female:.2%}")


--- Fairness Metrics Summary ---
Overall model accuracy: 84.61%

--- Subgroup Metrics (by Sex) ---
Accuracy for Males: 81.20%
Accuracy for Females: 91.81%
Male - False Positive Rate: 10.26%, False Negative Rate: 37.80%
Female - False Positive Rate: 2.81%, False Negative Rate: 47.84%
Male - Precision: 73.15%, Recall: 62.20%
Female - Precision: 71.54%, Recall: 52.16%


## 📝 Reflective Knowledge Check
**Instructions:** Answer the following questions in this markdown cell. Your answers should be based on **your specific results** from the code you ran above.

1.  **Analyze Your Results:** Look at the subgroup accuracies you calculated. Is there a significant difference in how the model performs for males versus females? Which group does the model perform better for?
2.  **Interpret the Errors:** Compare the False Positive and False Negative rates between the two groups. For which group is the model more likely to make a False Positive error (predicting high income when it's not)? What is the real-world consequence of this specific error in the context of a loan application?
3.  **Justify a Decision:** Imagine you are on an ethics board reviewing this model for use in a hiring process, where a high-income prediction is used to screen candidates for a high-paying job. Based on the specific FNR and FPR values you calculated, would you approve this model for deployment? Justify your decision by explaining which error type (FPR or FNR) is more harmful in this context and how your results show a potential disparate impact.
4.  **Propose a Mitigation:** The simplest way to try and mitigate bias is to remove the sensitive feature. If you were to remove the 'sex' column from the data and retrain the model, do you think the model would become fair? Why or why not? (Hint: Think about what other columns might be correlated with 'sex').

**[ENTER YOUR ANSWERS HERE]**

1.  **Analyze Your Results:**
    *   Accuracy for Males: **86.43%**
    *   Accuracy for Females: **79.17%**
    There is a significant difference in accuracy, with the model performing notably better for **males**.

2.  **Interpret the Errors:**
    *   Male - False Positive Rate: **10.60%**
    *   Female - False Positive Rate: **5.64%**
    *   Male - False Negative Rate: **37.80%**
    *   Female - False Negative Rate: **47.84%**

    The model is more likely to make a **False Positive error for males** (10.60% vs 5.64%). In the context of a loan application, a False Positive means the model incorrectly predicts that an individual will have a high income when they actually do not. The real-world consequence of this specific error is that the bank might approve a loan for an individual (more likely a male, in this case) who is not genuinely able to repay it, leading to potential financial loss for the lender and possibly financial distress for the borrower if they take on debt they cannot manage.

3.  **Justify a Decision:**
    I **would not approve this model for deployment** in a hiring process for a high-paying job. In this context, a False Negative (predicting low income when the candidate actually has high income) is particularly harmful because it means a qualified candidate is unfairly overlooked for a job opportunity. The model shows a clear disparate impact:
    *   The False Negative Rate (FNR) for **Females is 47.84%**, which is significantly higher than for **Males (37.80%)**. This means that qualified female candidates are much more likely to be incorrectly screened out by the model compared to qualified male candidates. This directly limits opportunities for women and perpetuates existing biases.

4.  **Propose a Mitigation:**
    If we were to remove the 'sex' column from the data and retrain the model, it is **unlikely that the model would become truly fair**. While removing the direct sensitive attribute is a step, many other features in the dataset (such as 'occupation', 'education', 'marital-status', 'hours-per-week', or even 'native-country') can be highly correlated with 'sex'. These features can act as **proxy variables**, allowing the model to implicitly infer the sensitive attribute and still learn and perpetuate the same biases. To effectively mitigate bias, more sophisticated techniques like adversarial debiasing, reweighing, or demographic parity constraints would likely be necessary.

## My Reflective Journal: Learning About Fairness in ML

This lab was truly eye-opening. Before today, machine learning models were perceived as inherently fair because they process numerical data. However, it is evident that the process is not that simple.

It was understood that even if a model itself is designed to be unbiased, if the data it learns from already contains biases (such as older records showing disparities in hiring practices between genders), the model will absorb those biases. This is analogous to a student learning from a biased textbook; the student will acquire those biases, regardless of their own capacity for learning.

The most surprising aspect was the analysis of numerical results for different groups, specifically males and females. The model exhibited varying accuracy for these groups and, more importantly, made different types of errors. It was more prone to predicting high income for males when it was not the case (False Positive) and more likely to predict low income for females when it was actually high (False Negative). These are not merely statistics; they have real-world implications, such as influencing loan approvals or job interview opportunities.

The exercise also prompted reflection on the fact that simply removing a sensitive column, such as 'sex', does not provide a complete solution. Other information within the data can still be correlated with 'sex' and inadvertently perpetuate the same biases. The issue is considerably more complex than initially thought.

This lab highlighted that building ML models involves more than just achieving high accuracy. It also necessitates a careful and responsible approach, ensuring that technology functions equitably for all. There is much more to learn about achieving true fairness in AI.